# 01 · Schema & data dictionary — Team 10 (Saideep Katragadda)

**Deliverable 3, Part 1.** This notebook:

1. Loads the dataset through the shared `load_data()` (from `load_data.py`), the same entry point every team-10 notebook uses.
2. Records the **shape**, each column's **type and meaning**, and its **number of distinct values**.
3. Runs a few **schema consistency checks**. These are rules the `.md` claims, such as `days_to_departure = departure_date − search_date`, tested against the actual rows.
4. Lists the **column names / meanings that need a PM's answer** instead of a guess.
5. Writes everything to `team-10-modeling/results/data_dictionary.md`.

> **NDA:** the data stays in the git-ignored `data/raw/`. The dictionary written by this notebook holds **counts and metadata only**: no row values, no sample rows, no min/max. **Clear outputs before committing** (Kernel → Restart & Clear Output).

Cleaning (duplicates, outliers), distributions, and leakage analysis belong to the later team notebooks. This one only describes the data.

In [ ]:
from pathlib import Path
import re

import pandas as pd

from load_data import load_data, RAW_DIR, REPO_ROOT, TARGET

RESULTS_DIR = REPO_ROOT / "team-10-modeling" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# The dataset's documentation file, which sits next to the CSV in data/raw/.
# When the real AA file arrives, point this at its .md (or leave None to auto-pick the only .md there).
DICT_MD_NAME = None
DATA_FILE_NAME = None   # None -> the single CSV in data/raw/

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

## 1. Load the data (via the shared loader)

In [ ]:
df = load_data(DATA_FILE_NAME)
raw = load_data(DATA_FILE_NAME, raw=True)   # untyped copy, used only to show what fix_types changed

n_rows, n_cols = df.shape
print(f"Rows:    {n_rows:,}")
print(f"Columns: {n_cols}")
print(f"Target:  {TARGET}")

In [ ]:
# What the loader's type fixes did: pandas' own guess vs. the team type
inferred = pd.read_csv(next(RAW_DIR.glob(DATA_FILE_NAME or "*.csv")), nrows=5000).dtypes.astype(str)
pd.DataFrame({"pandas_inferred": inferred, "after_load_data": df.dtypes.astype(str)})

## 2. Read the column descriptions from the dataset's `.md`

The `.md` that ships with the dataset has a `| Column | Type | Description |` table. We parse that table rather than retyping it, so the dictionary always matches the source document.

In [ ]:
if DICT_MD_NAME:
    dict_md_path = RAW_DIR / DICT_MD_NAME
else:
    mds = sorted(RAW_DIR.glob("*.md"))
    assert len(mds) == 1, f"Expected one .md in {RAW_DIR}, found {[p.name for p in mds]} - set DICT_MD_NAME."
    dict_md_path = mds[0]
print("Documentation file:", dict_md_path.name)

ROW_RE = re.compile(r"^\|\s*`([^`]+)`\s*\|\s*([^|]+?)\s*\|\s*(.+?)\s*\|\s*$")
doc_rows = [m.groups() for line in dict_md_path.read_text(encoding="utf-8").splitlines()
            if (m := ROW_RE.match(line.strip()))]
doc = pd.DataFrame(doc_rows, columns=["column", "documented_type", "meaning"]).set_index("column")
print(f"{len(doc)} columns documented")
doc

## 3. Column coverage: documented vs. actually in the file

In [ ]:
in_file, in_doc = set(df.columns), set(doc.index)
undocumented = sorted(in_file - in_doc)   # in the data but nobody told us what it means
absent = sorted(in_doc - in_file)         # documented but not delivered
print("In file but NOT documented:", undocumented or "none")
print("Documented but NOT in file:", absent or "none")

## 4. Per-column profile

For each column: the team dtype (after `load_data`), the documented type and meaning, the number of distinct values, and missingness. Distinct counts **exclude** missing values. `n_unique_raw` counts the untouched strings in the CSV file, which shows whether the type fix merged anything (e.g. `" JFK"` vs `"JFK"`).

In [ ]:
def role(col):
    if col == TARGET:
        return "target"
    if col in ("base_fare", "taxes_fees"):
        return "LEAKAGE (components of target)"
    if col == "search_id":
        return "id"
    return "feature"

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(dropna=True),
    "n_unique_raw": raw.replace("", pd.NA).nunique(dropna=True),
    "n_missing": df.isna().sum(),
})
profile["pct_missing"] = (profile["n_missing"] / n_rows * 100).round(2)
profile["pct_unique"] = (profile["n_unique"] / n_rows * 100).round(2)
profile["role"] = [role(c) for c in profile.index]
profile = profile.join(doc, how="left")
profile["meaning"] = profile["meaning"].fillna("**UNDOCUMENTED — ask PM**")
profile["documented_type"] = profile["documented_type"].fillna("?")

# Keep the file's column order
profile = profile.loc[df.columns, ["dtype", "documented_type", "role", "n_unique", "n_unique_raw",
                                   "n_missing", "pct_missing", "pct_unique", "meaning"]]
profile

In [ ]:
# Quick reads off the profile
print("Constant columns (1 distinct value -> no signal):", profile.index[profile.n_unique <= 1].tolist())
print("Columns with missing values:", profile.index[profile.n_missing > 0].tolist())
cat_like = profile[profile.dtype.isin(["category", "str", "object"])]
print("\nCategorical cardinality:")
print(cat_like["n_unique"].sort_values(ascending=False).to_string())
print("\nRoute (origin-destination) pairs:",
      df.groupby(["origin", "destination"], observed=True).ngroups)

## 5. Schema consistency checks

The `.md` makes claims about how columns relate to each other. Each one is tested below and reported as the number of rows that **break** the rule. These counts go into the dictionary, and any surprising result goes on the PM question list.

In [ ]:
def has(*cols):
    return all(c in df for c in cols)

checks = []
def check(name, cols, violations_fn):
    if has(*cols):
        n = int(violations_fn())
        checks.append({"rule": name, "violating_rows": n, "pct": round(n / n_rows * 100, 2)})
    else:
        checks.append({"rule": name, "violating_rows": None, "pct": None})

check("days_to_departure == departure_date - search_date",
      ["days_to_departure", "departure_date", "search_date"],
      lambda: ((df.departure_date - df.search_date).dt.days != df.days_to_departure).sum())
check("days_to_departure >= 0", ["days_to_departure"],
      lambda: (df.days_to_departure < 0).sum())
check("day_of_week matches departure_date", ["day_of_week", "departure_date"],
      lambda: (df.departure_date.dt.strftime("%a") != df.day_of_week.astype(str)).sum())
check("month matches departure_date", ["month", "departure_date"],
      lambda: (df.departure_date.dt.month != df.month).sum())
check("is_weekend_departure == (Fri/Sat/Sun)", ["is_weekend_departure", "departure_date"],
      lambda: ((df.departure_date.dt.dayofweek >= 4) != df.is_weekend_departure).sum())
check("nonstop == (num_stops == 0)", ["nonstop", "num_stops"],
      lambda: ((df.num_stops == 0) != df.nonstop).sum())
check("distance_miles constant per origin-destination", ["origin", "destination", "distance_miles"],
      lambda: (df.groupby(["origin", "destination"], observed=True).distance_miles
                 .transform("nunique") > 1).sum())
check("load_factor within [0, 1]", ["load_factor"],
      lambda: (~df.load_factor.between(0, 1)).sum())
check("load_factor ~= 1 - seats_remaining/aircraft_capacity (±0.01)",
      ["load_factor", "seats_remaining", "aircraft_capacity"],
      lambda: ((1 - df.seats_remaining / df.aircraft_capacity - df.load_factor).abs() > 0.01).sum())
check("seats_remaining <= aircraft_capacity", ["seats_remaining", "aircraft_capacity"],
      lambda: (df.seats_remaining > df.aircraft_capacity).sum())
check("total_price == base_fare + taxes_fees (±$0.02)", ["total_price", "base_fare", "taxes_fees"],
      lambda: ((df.base_fare + df.taxes_fees - df.total_price).abs() > 0.02).sum())
check("total_price > 0", ["total_price"], lambda: (df.total_price <= 0).sum())
check("search_id is unique", ["search_id"], lambda: df.search_id.duplicated().sum())
check("no exact duplicate rows", list(df.columns[:1]), lambda: df.duplicated().sum())
check("origin != destination", ["origin", "destination"],
      lambda: (df.origin.astype(str) == df.destination.astype(str)).sum())

checks_df = pd.DataFrame(checks)
checks_df

In [ ]:
# Fare bucket vs cabin: in real airline pricing each fare letter belongs to one cabin.
# If every letter appears in every cabin, the column's meaning needs confirming.
fare_class_in_every_cabin = False
if has("fare_booking_class", "cabin"):
    fb = pd.crosstab(df.fare_booking_class, df.cabin)
    cabins_per_class = (fb > 0).sum(axis=1)
    fare_class_in_every_cabin = bool((cabins_per_class == fb.shape[1]).all())
    print("Cabins each fare class appears in:\n", cabins_per_class.to_string())
    print("Every fare class appears in every cabin:", fare_class_in_every_cabin)
    print("\nCabin labels in the data:", sorted(df.cabin.dropna().astype(str).unique()))

## 6. Columns that need a PM's clarification

The brief says: *"ask a PM/team lead if a column name is ambiguous — don't guess at a real dataset's schema."* Each question below came from the `.md`, from the checks above, or from a modeling decision that depends on the answer. The **timing** questions matter most: a column measured *after* the fare was quoted would be leakage.

In [ ]:
c = checks_df.set_index("rule")["violating_rows"]
n_id_dup, n_row_dup = c.get("search_id is unique"), c.get("no exact duplicate rows")
dup_note = ("all of them are exact duplicate rows" if n_id_dup == n_row_dup
            else f"only {n_row_dup} of them are exact duplicate rows, so some ids are reused for different quotes")
cabin_labels = sorted(df["cabin"].dropna().astype(str).unique()) if "cabin" in df else []

pm_questions = [
    ("load_factor, seats_remaining",
     "Measured at search_date (when the fare was quoted) or at departure / a later snapshot? If later, they "
     "are leakage. Also, load_factor ≈ 1 − seats_remaining/aircraft_capacity "
     f"(rows off by more than 0.01: {c.get('load_factor ~= 1 - seats_remaining/aircraft_capacity (±0.01)')}). Is one simply derived from the other?"),
    ("competitor_min_price",
     "Which competitors are included, and is it captured at the same moment, for the same cabin and the same route as the AA quote? "
     "Why is it missing (no competitor on the route vs. scrape failure)? The two causes need different imputation."),
    ("base_fare, taxes_fees",
     "Are they ever known before the total price is? The .md says total_price = base_fare + taxes_fees exactly, which makes "
     f"them leakage. {c.get('total_price == base_fare + taxes_fees (±$0.02)')} rows break that sum. Is the error in total_price or in the parts?"),
    ("total_price",
     "Units and scope: USD? Per passenger? One-way or round trip? Is it the lowest available fare or the fare actually sold?"),
    ("fare_booking_class",
     ("Every fare letter shows up in every cabin (e.g. 'Y' in Basic Economy). In real AA inventory a letter maps to one cabin. "
      if fare_class_in_every_cabin else "")
     + "What does this column represent here, and is it set before or at purchase?"),
    ("cabin",
     f"Labels in the data: {cabin_labels} (the .md writes 'Business-First'). "
     "Are Business and First intentionally merged (e.g. 2-cabin vs 3-cabin aircraft)?"),
    ("search_id",
     f"Is this meant to be a unique quote key? {n_id_dup} ids repeat; {dup_note}. "
     "Are repeated searches real events or ingestion duplicates?"),
    ("search_date / departure_date",
     "Time zone of the dates, and is search_date a date only or truncated from a timestamp? This matters for a time-based train/test split."),
    ("departure_hour",
     "Local time at the origin? For connecting itineraries, is it the first leg's departure?"),
    ("distance_miles",
     "Great-circle O&D distance even for connecting itineraries, or flown miles?"),
    ("is_holiday_period",
     "Exact date windows for 'Thanksgiving / late-Dec / July-4'. Are they defined on departure_date only?"),
    ("is_weekend_departure",
     "Confirm the Fri–Sun definition (some analyses use Sat–Sun)."),
    ("aircraft_capacity",
     "Total seats on the equipment, or seats in the quoted cabin only? For connections, which leg?"),
    ("carrier",
     "Always 'AA' in this file. Are partner / codeshare (e.g. AA-marketed, Envoy-operated) itineraries included or excluded?"),
]
for name in undocumented:
    pm_questions.append((name, "Column is in the file but NOT in the dataset's documentation. What is it?"))
for name in absent:
    pm_questions.append((name, "Documented but missing from the delivered file. Was it dropped on purpose?"))

pm_df = pd.DataFrame(pm_questions, columns=["column(s)", "question for PM"])
pm_df

## 7. Write `results/data_dictionary.md`

In [ ]:
def md_table(frame, index=True):
    frame = frame.reset_index() if index else frame
    frame = frame.astype(object).where(frame.notna(), "—")
    esc = lambda v: str(v).replace("|", "\\|").replace("\n", " ")
    header = "| " + " | ".join(esc(c) for c in frame.columns) + " |"
    sep = "|" + "|".join("---" for _ in frame.columns) + "|"
    body = ["| " + " | ".join(esc(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join([header, sep, *body])

col_table = profile.rename_axis("column")[
    ["dtype", "documented_type", "role", "n_unique", "n_missing", "pct_missing", "meaning"]
]

lines = [
    "# Data dictionary — Team 10 (Deliverable 3)",
    "",
    f"_Generated by `team-10-modeling/notebooks/01_schema_saideep.ipynb` from `data/raw/{dict_md_path.name}`"
    f" and the data file loaded by `load_data()`. Do not hand-edit; re-run the notebook._",
    "",
    "> **NDA:** this file holds schema metadata and counts only: no row values, samples or ranges."
    " Keep it that way.",
    "",
    "## Overview",
    "",
    f"- **Rows:** {n_rows:,}",
    f"- **Columns:** {n_cols}",
    f"- **Target:** `{TARGET}`",
    f"- **Grain (per the .md):** one fare search result (a priced itinerary at a point in time)",
    f"- **Exact duplicate rows:** {int(df.duplicated().sum()):,}",
    f"- **Columns with missing values:** "
    + (", ".join(f"`{k}` ({v}%)" for k, v in profile.loc[profile.n_missing > 0, "pct_missing"].items()) or "none"),
    f"- **Undocumented columns:** {', '.join(f'`{x}`' for x in undocumented) or 'none'}",
    f"- **Documented but absent:** {', '.join(f'`{x}`' for x in absent) or 'none'}",
    "",
    "## Columns",
    "",
    "`dtype` is the type after the shared `load_data()` type fixes. `n_unique` counts distinct non-missing values."
    " `role` flags the target, the id, and columns that leak the target.",
    "",
    md_table(col_table),
    "",
    "## Schema consistency checks",
    "",
    "Rows that break each rule the documentation implies (— = column not present).",
    "",
    md_table(checks_df, index=False),
    "",
    "## Needs PM clarification",
    "",
    "We are **not** guessing these. Answers change which columns are safe to use (leakage) and how they are cleaned.",
    "",
    md_table(pm_df, index=False),
    "",
]
out_path = RESULTS_DIR / "data_dictionary.md"
out_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {out_path.relative_to(REPO_ROOT)}  ({n_rows:,} rows x {n_cols} cols, {len(pm_df)} PM questions)")

## Takeaways for the rest of the team

- **Use `from load_data import load_data`** in every notebook. Dates, booleans, ordered `day_of_week`, and nullable ints are already typed.
- `load_data()` does **not** remove duplicates or outliers and does **not** drop `base_fare` / `taxes_fees`. Each later notebook makes those decisions explicitly and gives the reason.
- Before building features on `load_factor`, `seats_remaining`, or `competitor_min_price`, check the PM answers on **when** they were measured.
- When the real AA file lands in `data/raw/`, re-run this notebook first. Any schema change will show up in the coverage section (§3) and in the loader's warnings.